# Controlled 2x6 Factorial Benchmark Pipeline
## A Data-Centric Benchmark for Multi-Class Power-Line Defect Detection
**Authors:** Phạm Hoàng Minh Châu, Đỗ Trung Quân, Nguyễn Nhật Thông  
**Mentor:** Đặng Ngọc Minh Đức  

This notebook orchestrates the end-to-end benchmark workflow:
1. **Environment Setup & GPU Verification (RTX 4060)**
2. **Dataset Configuration:** Raw Combined (V6.2E) vs Curated Combined (8-rule curation)
3. **Resumable Training Runner:** 12 experimental variants (2 datasets × 6 models)
4. **Evaluation on Frozen Test Set:** Accuracy metrics + Hardware latency/FPS profiling
5. **Factorial Analysis & Reporting:** Data Curation Gain vs Model Scaling Gain

In [1]:
# Cell 1: Environment Setup & Module Imports
import sys, os, time
from pathlib import Path
import torch
import pandas as pd
import matplotlib.pyplot as plt

# Ensure local src is in path
CURRENT_DIR = Path.cwd()
if (CURRENT_DIR / "benchmakring_transmission_line_defect_model").exists():
    CURRENT_DIR = CURRENT_DIR / "benchmakring_transmission_line_defect_model"
    os.chdir(CURRENT_DIR)
if str(CURRENT_DIR) not in sys.path:
    sys.path.insert(0, str(CURRENT_DIR))

from src.config import get_benchmark_config, get_class_names
from src.dataset import setup_dataset_yaml, verify_dataset_structure
from src.trainer import BenchmarkTrainer
from src.evaluator import BenchmarkEvaluator
from src.report import BenchmarkReporter

print("=== Environment Verification ===")
print("PyTorch Version:", torch.__version__)
print("CUDA Available :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device Name:", torch.cuda.get_device_name(0))
    print("Total VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2))
print("Classes (6-class taxonomy):", get_class_names())

=== Environment Verification ===
PyTorch Version: 2.14.1+cu126
CUDA Available : True
GPU Device Name: NVIDIA GeForce RTX 4060 Laptop GPU
Total VRAM (GB): 8.0
Classes (6-class taxonomy): {0: 'insulator', 1: 'insulator_broken', 2: 'insulator_flashover', 3: 'bird_nest', 4: 'foreign_object', 5: 'broken_strand'}


In [2]:
# Cell 2: Dataset Verification (Raw Combined vs Curated Combined)
cfg = get_benchmark_config()
raw_dir = CURRENT_DIR / "data" / "raw_combined"
curated_dir = CURRENT_DIR / "data" / "curated_combined"

raw_stats = verify_dataset_structure(raw_dir, "val")
curated_stats = verify_dataset_structure(curated_dir, "val_clean")

stats_df = pd.DataFrame([
    {"Dataset": "Raw Combined (V6.2E uncurated)", "Train Images": raw_stats["train"]["images"], "Train Labels": raw_stats["train"]["labels"], "Val/Val_clean": raw_stats["val"]["images"], "YAML": cfg["datasets"]["raw"]["yaml_path"]},
    {"Dataset": "Curated Combined (8-rule curation)", "Train Images": curated_stats["train"]["images"], "Train Labels": curated_stats["train"]["labels"], "Val/Val_clean": curated_stats["val"]["images"], "YAML": cfg["datasets"]["curated"]["yaml_path"]}
])
print("=== Dataset Verification Status ===")
display(stats_df)

=== Dataset Verification Status ===


,Dataset,Train Images,Train Labels,Val/Val_clean,YAML
0,Raw Combined (V6.2E uncurated),18334,18334,2272,configs/dataset_raw.yaml
1,Curated Combined (8-rule curation),17903,17903,2271,configs/dataset_curated.yaml


In [3]:
# Cell 3: Training Management (Fully Resumable 2x6 Factorial)
trainer = BenchmarkTrainer()

# Display current status of all 12 runs
status_table = pd.DataFrame(list(trainer.get_run_status().items()), columns=["Run Key", "Status"])
display(status_table)

# To train a single variant:
# trainer.train_single_run(dataset_type="raw", model_key="yolo11n")

# To launch the full benchmark suite (automatically resumes if stopped!)    :
trainer.run_all_benchmarks(datasets=["raw", "curated"], models=["yolo11n", "yolo11m", "yolo26n", "yolo26m"])

,Run Key,Status
0,raw_yolo11n,COMPLETED
1,raw_yolo11m,COMPLETED
2,raw_yolo26n,COMPLETED
3,raw_yolo26m,COMPLETED
4,raw_rtdetr_r50,PENDING
5,raw_rtdetr_r101,PENDING
6,curated_yolo11n,COMPLETED
7,curated_yolo11m,COMPLETED
8,curated_yolo26n,COMPLETED
9,curated_yolo26m,COMPLETED



>>> Launching Factorial Benchmark: 2 datasets x 4 models = 8 runs <<<
[SKIP] Run 'raw_yolo11n' already completed. Using saved checkpoint.
[SKIP] Run 'raw_yolo11m' already completed. Using saved checkpoint.
[SKIP] Run 'raw_yolo26n' already completed. Using saved checkpoint.
[SKIP] Run 'raw_yolo26m' already completed. Using saved checkpoint.
[SKIP] Run 'curated_yolo11n' already completed. Using saved checkpoint.
[SKIP] Run 'curated_yolo11m' already completed. Using saved checkpoint.
[SKIP] Run 'curated_yolo26n' already completed. Using saved checkpoint.
[SKIP] Run 'curated_yolo26m' already completed. Using saved checkpoint.


{'raw_yolo11n': {'run_key': 'raw_yolo11n',
  'dataset': 'raw',
  'model': 'yolo11n',
  'family': 'yolo11',
  'status': 'completed',
  'best_weights': 'D:\\Data-Centric-UAV-Powerline-Defect-Detection\\benchmakring_transmission_line_defect_model\\runs\\raw_yolo11n\\weights\\best.pt',
  'last_weights': 'D:\\Data-Centric-UAV-Powerline-Defect-Detection\\benchmakring_transmission_line_defect_model\\runs\\raw_yolo11n\\weights\\last.pt',
  'run_dir': 'D:\\Data-Centric-UAV-Powerline-Defect-Detection\\benchmakring_transmission_line_defect_model\\runs\\raw_yolo11n',
  'training_time_seconds': 7689.53542470932,
  'peak_vram_mb': 2143.17578125,
  'completed_at': '2026-10-21 02:52:56'},
 'raw_yolo11m': {'run_key': 'raw_yolo11m',
  'dataset': 'raw',
  'model': 'yolo11m',
  'family': 'yolo11',
  'status': 'completed',
  'best_weights': 'D:\\Data-Centric-UAV-Powerline-Defect-Detection\\benchmakring_transmission_line_defect_model\\runs\\raw_yolo11m\\weights\\best.pt',
  'last_weights': 'D:\\Data-Centric

In [ ]:
# Cell 4: Evaluate RT-DETR 50/100 raw and curated models on V6.2E test
evaluator = BenchmarkEvaluator()

# Both raw and curated checkpoints are evaluated on the same independent test set.
evaluation_yaml = r"D:\Data-Centric-UAV-Powerline-Defect-Detection\Uav-Powerline-Defect-Detection\data\working\v6p2e_test\dataset.yaml"
# Ultralytics requires train and val keys even when evaluating split="test".
import yaml
with evaluation_yaml.open("r", encoding="utf-8") as f:
    evaluation_data = yaml.safe_load(f)
if not evaluation_data.get("train"):
    evaluation_data["train"] = evaluation_data["test"]
    with evaluation_yaml.open("w", encoding="utf-8") as f:
        yaml.safe_dump(evaluation_data, f, sort_keys=False)

checkpoint_paths = {
    "curated_rtdetr50": CURRENT_DIR / "runs" / "curated_rtdetr50" / "weights" / "last.pt",
    "curated_rtdetr100": CURRENT_DIR / "runs" / "curated_rtdetr100" / "weights" / "last.pt",
    "raw_rtdetr50": CURRENT_DIR / "runs" / "raw_rtdetr50" / "weights" / "last.pt",
    "raw_rtdetr100": CURRENT_DIR / "runs" / "raw_rtdetr100" / "weights" / "last.pt",
}
missing_checkpoints = [str(path) for path in checkpoint_paths.values() if not path.is_file()]
if missing_checkpoints:
    raise FileNotFoundError("Missing RT-DETR checkpoints:\n" + "\n".join(missing_checkpoints))

selected_runs = {
    run_key: {"last_weights": str(checkpoint_path)}
    for run_key, checkpoint_path in checkpoint_paths.items()
}
split_by_dataset = {"raw": "test", "curated": "test"}

print(f"Evaluating four RT-DETR checkpoints on V6.2E test set: {evaluation_yaml}")
df_summary = evaluator.evaluate_all(
    selected_runs,
    evaluation_yaml,
    split_by_dataset=split_by_dataset,
    checkpoint_field="last_weights"
)
if not df_summary.empty:
    display(df_summary)
else:
    print("No completed models found to evaluate. Run training first.")

In [ ]:
# Cell 5: 2x6 Factorial Matrix, Visualization & Report Generation
reporter = BenchmarkReporter()
report_md = reporter.generate_report()

# Load matrix for visual plotting
matrix_csv = CURRENT_DIR / "results" / "factorial_benchmark_matrix.csv"
if matrix_csv.exists():
    df_mat = pd.read_csv(matrix_csv)
    display(df_mat)
    
    # Bar chart: Raw vs Curated mAP@0.5
    if "Raw mAP50" in df_mat.columns and "Curated mAP50" in df_mat.columns:
        plt.figure(figsize=(10, 5))
        x = range(len(df_mat))
        # Convert numeric if available
        try:
            raw_vals = pd.to_numeric(df_mat["Raw mAP50"], errors='coerce')
            cur_vals = pd.to_numeric(df_mat["Curated mAP50"], errors='coerce')
            plt.bar([i - 0.2 for i in x], raw_vals, width=0.4, label="Raw Combined", color="salmon")
            plt.bar([i + 0.2 for i in x], cur_vals, width=0.4, label="Curated Combined", color="teal")
            plt.xticks(x, df_mat["Architecture"])
            plt.ylabel("mAP @ 0.5")
            plt.title("Controlled 2x6 Factorial: Raw vs Curated Data Performance")
            plt.legend()
            plt.grid(axis="y", linestyle="--", alpha=0.7)
            plt.show()
        except Exception as e:
            print("Plotting error:", e)

print("Report saved in: results/factorial_benchmark_report.md")